In [0]:
from pyspark.sql.functions import *

In [0]:
#Read Data 
customers = spark.read.option("header", True).option("inferSchema", True) \
    .csv("/Volumes/workspace/default/retail_data/clean/customers")

orders = spark.read.option("header", True).option("inferSchema", True) \
    .csv("/Volumes/workspace/default/retail_data/clean/orders")

payments = spark.read.option("header", True).option("inferSchema", True) \
    .csv("/Volumes/workspace/default/retail_data/clean/payments")

products = spark.read.option("header", True).option("inferSchema", True) \
    .csv("/Volumes/workspace/default/retail_data/clean/products")


In [0]:
#Mask customer id, PII masked, dob to age bands
customers_secure = customers \
    .withColumn("customer_token", sha2(col("customer_id"), 256)) \
    .withColumn("customer_name", concat(substring("customer_name", 1, 1), lit("***"))) \
    .withColumn("email", concat(substring("email", 1, 1), lit("***@"), element_at(split("email", "@"), -1))) \
    .withColumn("phone", concat(lit("(XXX) XXX-"), substring("phone", -4, 4))) \
    .withColumn("address", concat(lit("*** "), element_at(split("address", " "), -1))) \
    .withColumn("age", floor(months_between(current_date(), to_date("dob")) / 12).cast("int")) \
    .withColumn("age_band",
        when(col("age") < 25, "18-24")
        .when(col("age") < 35, "25-34")
        .when(col("age") < 45, "35-44")
        .when(col("age") < 55, "45-54")
        .otherwise("55+")) \
    .select("customer_name", "email", "phone", "address", "city", "state",
            "signup_date", "customer_token", "age", "age_band")


In [0]:
#Orders: swap customer_id for the same token (sha2 is deterministic, so it matches customers)
orders_clean = orders \
    .withColumn("customer_token", sha2(col("customer_id"), 256)) \
    .select("order_id", "product_id", "quantity", "unit_price", "total_amount",
            "order_date", "status", "payment_method", "customer_token")

In [0]:
#Drop cvv, mask PCI, price buckets
payments_secure = payments \
    .drop("cvv") \
    .withColumn("card_number", concat(lit("*" * 12), substring(col("card_number").cast("string"), -4, 4))) \
    .withColumn("spending_bucket",
        when(col("amount") < 1000, "Low")
        .when(col("amount") < 5000, "Medium")
        .when(col("amount") < 15000, "High")
        .otherwise("Very High")) \
    .select("payment_id", "order_id", "card_number", "amount",
            "payment_date", "payment_status", "spending_bucket")

In [0]:
#Save to silver layer
silver_path = "/Volumes/workspace/default/retail_data/silver/"

customers_secure.write.mode("overwrite").parquet(silver_path + "customers_secure.parquet")
orders_clean.write.mode("overwrite").parquet(silver_path + "orders_clean.parquet")
payments_secure.write.mode("overwrite").parquet(silver_path + "payments_secure.parquet")
products.write.mode("overwrite").parquet(silver_path + "products_clean.parquet")

['order_id', 'product_id', 'quantity', 'unit_price', 'total_amount', 'order_date', 'status', 'payment_method', 'customer_token']
